<a href="https://colab.research.google.com/github/sYanXO/LLM-inferencing/blob/main/tinyllama_batch_processing_stress_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import numpy as np
import traceback

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print()

# Load model and tokenizer
print("Loading TinyLlama...")
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map=device
)
tokenizer.pad_token = tokenizer.eos_token
print("Model loaded!\n")

# Base prompts
base_prompts = [
    "What is machine learning?",
    "Explain quantum computing in simple terms.",
    "How do neural networks work?",
    "What is the future of AI?",
    "Describe the importance of data privacy.",
    "What are transformers in deep learning?",
    "Explain backpropagation.",
    "What is reinforcement learning?",
]

# ============================================================================
# PHASE 1: PUSH TO BREAKING POINT
# ============================================================================
print("="*80)
print("PHASE 1: PUSHING TO THE BREAKING POINT")
print("="*80)

batch_sizes = [32, 64, 128, 256, 512, 1024]
breaking_point = None
max_successful_batch = 32

results = {
    "batch_size": [],
    "pp_throughput": [],
    "tg_throughput": [],
    "gpu_memory": [],
    "status": [],
    "error": []
}

for batch_size in batch_sizes:
    print(f"\n{'='*80}")
    print(f"Testing Batch Size: {batch_size}")
    print(f"{'='*80}")

    # Create prompts
    prompts = [base_prompts[i % len(base_prompts)] for i in range(batch_size)]

    try:
        # Tokenize
        batch_inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        batch_input_ids = batch_inputs["input_ids"]
        batch_attention_mask = batch_inputs["attention_mask"]
        input_length = batch_input_ids.shape[1]

        print(f"Batch shape: {batch_input_ids.shape}")
        print(f"Input tokens per prompt (padded): {input_length}")

        # Measure Prompt Processing (PP)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        pp_start = time.time()

        with torch.no_grad():
            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                output_hidden_states=False
            )

        torch.cuda.synchronize()
        pp_time = time.time() - pp_start

        # Measure Token Generation (TG)
        torch.cuda.synchronize()
        tg_start = time.time()

        with torch.no_grad():
            generated = model.generate(
                batch_input_ids,
                attention_mask=batch_attention_mask,
                max_new_tokens=50,
                min_new_tokens=50,
                do_sample=False,
                temperature=None,
                top_p=None
            )

        torch.cuda.synchronize()
        tg_time = time.time() - tg_start

        gpu_memory = torch.cuda.max_memory_allocated() / (1024**3)

        # Calculate throughputs
        total_pp_tokens = input_length * batch_size
        total_tg_tokens = 50 * batch_size

        pp_throughput = total_pp_tokens / pp_time
        tg_throughput = total_tg_tokens / tg_time

        print(f"✓ SUCCESS")
        print(f"PP time: {pp_time:.4f}s ({pp_throughput:.2f} tokens/sec)")
        print(f"TG time: {tg_time:.4f}s ({tg_throughput:.2f} tokens/sec)")
        print(f"Peak GPU memory: {gpu_memory:.2f} GB")

        results["batch_size"].append(batch_size)
        results["pp_throughput"].append(pp_throughput)
        results["tg_throughput"].append(tg_throughput)
        results["gpu_memory"].append(gpu_memory)
        results["status"].append("✓ SUCCESS")
        results["error"].append(None)

        max_successful_batch = batch_size

    except RuntimeError as e:
        error_msg = str(e)[:150]
        print(f"❌ FAILED")
        print(f"Error: {error_msg}")

        if "CUDA out of memory" in error_msg or "OutOfMemoryError" in error_msg:
            print(f"⚠️  OUT OF MEMORY - Breaking point found!")
            breaking_point = batch_size
            results["batch_size"].append(batch_size)
            results["pp_throughput"].append(0)
            results["tg_throughput"].append(0)
            results["gpu_memory"].append(None)
            results["status"].append("❌ OOM")
            results["error"].append(error_msg)
            break
        else:
            results["batch_size"].append(batch_size)
            results["pp_throughput"].append(0)
            results["tg_throughput"].append(0)
            results["gpu_memory"].append(None)
            results["status"].append("❌ ERROR")
            results["error"].append(error_msg)
            break

    except Exception as e:
        print(f"❌ UNEXPECTED ERROR: {type(e).__name__}")
        print(f"{str(e)[:150]}")
        break

print(f"\n{'='*80}")
print(f"PHASE 1 SUMMARY: Max successful batch = {max_successful_batch}")
print(f"Breaking point: {breaking_point if breaking_point else 'Not found (still scaling!)'}")
print(f"{'='*80}")

# ============================================================================
# PHASE 2: HACK ATTEMPT 1 - GRADIENT CHECKPOINTING
# ============================================================================
print(f"\n\n{'='*80}")
print("PHASE 2A: HACK #1 - GRADIENT CHECKPOINTING")
print("="*80)
print("(Reduces memory by trading compute for memory)")

try:
    # Enable gradient checkpointing
    model.gradient_checkpointing_enable()
    print("✓ Gradient checkpointing enabled")

    # Try the breaking batch size
    test_batch = breaking_point if breaking_point else max_successful_batch * 2
    print(f"\nTrying batch size {test_batch} with gradient checkpointing...")

    prompts = [base_prompts[i % len(base_prompts)] for i in range(test_batch)]
    batch_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        generated = model.generate(
            batch_inputs["input_ids"],
            attention_mask=batch_inputs["attention_mask"],
            max_new_tokens=50,
            min_new_tokens=50,
            do_sample=False
        )

    torch.cuda.synchronize()
    elapsed = time.time() - start
    gpu_mem = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"✓ SUCCESS with gradient checkpointing!")
    print(f"Batch size: {test_batch}")
    print(f"Time: {elapsed:.4f}s")
    print(f"GPU memory: {gpu_mem:.2f} GB")
    print(f"TG throughput: {(test_batch * 50) / elapsed:.2f} tokens/sec")

except Exception as e:
    print(f"❌ Failed: {str(e)[:100]}")
    model.gradient_checkpointing_enable(False)

# ============================================================================
# PHASE 2B: HACK ATTEMPT 2 - 8-BIT QUANTIZATION
# ============================================================================
print(f"\n\n{'='*80}")
print("PHASE 2B: HACK #2 - 8-BIT QUANTIZATION (bitsandbytes)")
print("="*80)
print("(Reduces memory by using 8-bit instead of 16-bit)")

try:
    print("Installing bitsandbytes...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=False)

    from transformers import BitsAndBytesConfig

    print("Reloading model with 8-bit quantization...")
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    model_8bit = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map=device
    )

    test_batch = breaking_point if breaking_point else max_successful_batch * 2
    print(f"\nTrying batch size {test_batch} with 8-bit quantization...")

    prompts = [base_prompts[i % len(base_prompts)] for i in range(test_batch)]
    batch_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        generated = model_8bit.generate(
            batch_inputs["input_ids"],
            attention_mask=batch_inputs["attention_mask"],
            max_new_tokens=50,
            min_new_tokens=50,
            do_sample=False
        )

    torch.cuda.synchronize()
    elapsed = time.time() - start
    gpu_mem = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"✓ SUCCESS with 8-bit quantization!")
    print(f"Batch size: {test_batch}")
    print(f"Time: {elapsed:.4f}s")
    print(f"GPU memory: {gpu_mem:.2f} GB")
    print(f"TG throughput: {(test_batch * 50) / elapsed:.2f} tokens/sec")

except Exception as e:
    print(f"❌ Failed: {str(e)[:100]}")

# ============================================================================
# PHASE 2C: HACK ATTEMPT 3 - SEQUENTIAL BATCH PROCESSING
# ============================================================================
print(f"\n\n{'='*80}")
print("PHASE 2C: HACK #3 - SEQUENTIAL MINI-BATCHING")
print("="*80)
print("(Process large batch as multiple smaller batches, collect results)")

try:
    # Reset to original float16 model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map=device
    )

    test_batch = breaking_point if breaking_point else max_successful_batch * 2
    mini_batch_size = max_successful_batch

    print(f"\nProcessing {test_batch} prompts as {test_batch // mini_batch_size} mini-batches of {mini_batch_size}")

    prompts = [base_prompts[i % len(base_prompts)] for i in range(test_batch)]
    all_outputs = []

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        for i in range(0, test_batch, mini_batch_size):
            batch_prompts = prompts[i:i+mini_batch_size]
            batch_inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            generated = model.generate(
                batch_inputs["input_ids"],
                attention_mask=batch_inputs["attention_mask"],
                max_new_tokens=50,
                min_new_tokens=50,
                do_sample=False
            )
            all_outputs.append(generated)

            # Clear cache between batches
            torch.cuda.empty_cache()

    torch.cuda.synchronize()
    elapsed = time.time() - start
    gpu_mem = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"✓ SUCCESS with sequential mini-batching!")
    print(f"Total batch size: {test_batch}")
    print(f"Mini-batch size: {mini_batch_size}")
    print(f"Time: {elapsed:.4f}s")
    print(f"GPU memory: {gpu_mem:.2f} GB")
    print(f"TG throughput: {(test_batch * 50) / elapsed:.2f} tokens/sec")

except Exception as e:
    print(f"❌ Failed: {str(e)[:100]}")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print(f"\n\n{'='*80}")
print("FINAL SUMMARY - Phase 1 Results")
print(f"{'='*80}")
print(f"{'Batch':<10} {'TG Throughput':<18} {'GPU Mem':<12} {'Status':<20}")
print(f"{'-'*80}")

for i, bs in enumerate(results["batch_size"]):
    tg = results["tg_throughput"][i]
    mem = results["gpu_memory"][i]
    status = results["status"][i]

    mem_str = f"{mem:.2f} GB" if mem is not None else "N/A"
    print(f"{bs:<10} {tg:<18.2f} {mem_str:<12} {status:<20}")

print(f"\n{'='*80}")
print("INSIGHTS")
print(f"{'='*80}")
print(f"✓ Max successful batch (Phase 1): {max_successful_batch}")
print(f"✓ Breaking point: {breaking_point if breaking_point else 'Not found!'}")
print(f"✓ Memory used at max: {results['gpu_memory'][len(results['gpu_memory'])-2] if len(results['gpu_memory']) > 1 else 'N/A'} GB (out of 16 GB)")
print(f"\n💡 Takeaway: Batching from 1 to {max_successful_batch} = {results['tg_throughput'][results['batch_size'].index(max_successful_batch)] / results['tg_throughput'][0]:.1f}x speedup!")
print(f"{'='*80}")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB

Loading TinyLlama...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Trying batch size 2048 with 8-bit quantization...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


✓ SUCCESS with 8-bit quantization!
Batch size: 2048
Time: 55.0820s
GPU memory: 10.84 GB
TG throughput: 1859.05 tokens/sec


PHASE 2C: HACK #3 - SEQUENTIAL MINI-BATCHING
(Process large batch as multiple smaller batches, collect results)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Processing 2048 prompts as 2 mini-batches of 1024
✓ SUCCESS with sequential mini-batching!
Total batch size: 2048
Mini-batch size: 1024
Time: 32.0242s
GPU memory: 6.04 GB
TG throughput: 3197.58 tokens/sec


FINAL SUMMARY - Phase 1 Results
Batch      TG Throughput      GPU Mem      Status              
--------------------------------------------------------------------------------
32         844.71             2.15 GB      ✓ SUCCESS           
64         2070.96            2.24 GB      ✓ SUCCESS           
128        3077.03            2.41 GB      ✓ SUCCESS           
256        3466.80            2.77 GB      ✓ SUCCESS           
512        3689.65            3.47 GB      ✓ SUCCESS           
1024       3697.47            4.88 GB      ✓ SUCCESS           

INSIGHTS
✓ Max successful batch (Phase 1): 1024
✓ Breaking point: Not found!
✓ Memory used at max: 3.469426155090332 GB (out of 16 GB)

💡 Takeaway: Batching from 1 to 1024 = 4.4x speedup!
